In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Schema definition & read csv
from pyspark.sql.types import *
from pyspark.sql.functions import when, lit, col, current_timestamp, input_file_name
    
# Create the schema for the table
orderSchema = StructType([
    StructField("deck", StringType()),
    StructField("maindeck", StringType()),
    StructField("count", IntegerType()),
    StructField("card", StringType())
    ])
    
# Import all files from bronze folder of lakehouse
df = spark.read.format("csv").option("header", "true").option("delimiter", ";").schema(orderSchema).load("Files/decks_bronze/*.csv")

    
# Add columns IsFlagged, CreatedTS and ModifiedTS
df = df.withColumn("Created_or_Modified_TS", current_timestamp())

# Display the first 10 rows of the dataframe to preview your data
display(df.head(10))

StatementMeta(, b1f82abc-abf9-41a4-8f12-178d99a5f573, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b7bcbb46-2937-4741-aa3a-b9ca61c34292)

In [10]:
# Define the schema for the sales_silver table
    
from pyspark.sql.types import *
from delta.tables import *
    
DeltaTable.createIfNotExists(spark) \
    .tableName("dbo.decks_silver") \
    .addColumn("deck", StringType()) \
    .addColumn("card", StringType()) \
    .addColumn("maindeck", StringType()) \
    .addColumn("count", IntegerType()) \
    .addColumn("Created_or_Modified_TS", TimestampType()) \
    .execute()

StatementMeta(, 5a6ca38c-2920-4d8b-8370-a2a6a69dfd0b, 12, Finished, Available, Finished, False)

In [14]:
display(df.head(100))

StatementMeta(, 5a6ca38c-2920-4d8b-8370-a2a6a69dfd0b, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe025f9f-2f32-4ca3-8008-c48f655e3f43)

In [2]:
# Update existing records and insert new ones based on a condition defined by the columns SalesOrderNumber, OrderDate, CustomerName, and Item.

from delta.tables import *
    
deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/decks_silver')
    
dfUpdates = df
    
deltaTable.alias('silver') \
  .merge(
    dfUpdates.alias('updates'),
    'silver.deck = updates.deck and silver.card = updates.card and silver.maindeck = updates.maindeck'
  ) \
   .whenMatchedUpdate(set =
    {
      "maindeck": "updates.maindeck",
      "count": "updates.count",
      "Created_or_Modified_TS": "updates.Created_or_Modified_TS" 
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "deck": "updates.deck",
      "card": "updates.card",
      "maindeck": "updates.maindeck",
      "count": "updates.count",
      "Created_or_Modified_TS": "updates.Created_or_Modified_TS"
    }
  ) \
  .whenNotMatchedBySourceDelete() \
  .execute()

StatementMeta(, b1f82abc-abf9-41a4-8f12-178d99a5f573, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.types import *
from delta.tables import *
    
# Create customer_gold dimension delta table
DeltaTable.createIfNotExists(spark) \
    .tableName("dbo.dimcard") \
    .addColumn("CardName", StringType()) \
    .addColumn("CardID", LongType()) \
    .execute()

StatementMeta(, 07b3473c-74d1-42aa-bf3d-29c550665cc3, 5, Finished, Available, Finished)

In [11]:
df = spark.read.table("dbo.decks_silver")


from pyspark.sql.functions import col, split
    
# Create customer_silver dataframe
    
dfdimCard_silver = df.dropDuplicates(["card"]).select(col("Card")) 
    
    
# Display the first 10 rows of the dataframe to preview your data

display(dfdimCard_silver.head(10))

StatementMeta(, 4450e732-4e81-4624-be47-637280d3e236, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a106c08e-df17-4993-a8b8-beb0ce9c9fae)

In [7]:
display(df.head(100))

StatementMeta(, d441ac95-2163-4972-a1d6-bd8206a51793, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d68f119-5e52-47ed-a440-5e936773d0de)

In [3]:
from pyspark.sql.functions import monotonically_increasing_id, col, when, coalesce, max, lit
    
dfdimCard_temp = spark.read.table("dbo.dimCard")
    
MAXCardID = dfdimCard_temp.select(coalesce(max(col("CardID")),lit(0)).alias("MAXCardID")).first()[0]
    
dfdimCard_gold = dfdimCard_silver.join(dfdimCard_temp,(dfdimCard_silver.Card == dfdimCard_temp.CardName) , "left_anti")
    
dfdimCard_gold = dfdimCard_gold.withColumn("CardID",monotonically_increasing_id() + MAXCardID + 1)

# Display the first 10 rows of the dataframe to preview your data

display(dfdimCard_gold.head(10))

StatementMeta(, 76d09bd9-8b02-4415-b638-3a996be2fa15, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, aed04834-6ccb-4c17-a760-59ca3d42da03)

In [9]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/dimcard')
    
dfUpdates = dfdimCard_gold
    
deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.CardName = updates.Card'
  ) \
   .whenMatchedUpdate(set =
    {
          
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "CardName": "updates.Card",
      "CardID": "updates.CardID"
    }
  ) \
  .execute()

StatementMeta(, 76d09bd9-8b02-4415-b638-3a996be2fa15, 11, Finished, Available, Finished)

In [2]:
from pyspark.sql.types import *
from delta.tables import *

## fact_decks create table

DeltaTable.createIfNotExists(spark) \
    .tableName("dbo.factdecks") \
    .addColumn("CardID", LongType()) \
    .addColumn("deck", StringType()) \
    .addColumn("card", StringType()) \
    .addColumn("maindeck", StringType()) \
    .addColumn("count", IntegerType()) \
    .execute()

StatementMeta(, 44e2b0d1-5d22-41f6-8ad7-348b73966e98, 4, Finished, Available, Finished, False)

In [1]:
# Load data to the dataframe as a starting point to create the gold layer
df = spark.read.table("dbo.decks_silver")

dfdimcard_temp = spark.read.table("dbo.dimCard")

dffact_decks = df.alias("df1").join(dfdimcard_temp.alias("df2"),(df.card == dfdimcard_temp.CardName))
#display(dfdimcard_temp.head(10))
#display(df.head(10))
display(dffact_decks.head(10))

StatementMeta(, c3b96023-c44e-40fc-9cf0-f0fd4a8aad4d, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d272e710-3368-488e-af7c-759d399eac78)

In [2]:
from delta.tables import *
    
deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/factdecks')
            
dfUpdates = dffact_decks 
            
deltaTable.alias('gold') \
  .merge(
        dfUpdates.alias('updates'),
        'gold.deck = updates.deck AND gold.card = updates.card'
        ) \
.whenMatchedUpdate(set =
    {
          
    }
  ) \
.whenNotMatchedInsert(values =
         { 
           "CardID": "updates.CardID",
           "deck": "updates.deck",
           "card": "updates.card",
           "maindeck": "updates.maindeck",
           "count": "updates.count"
          }
          ) \
          .execute()

StatementMeta(, c3b96023-c44e-40fc-9cf0-f0fd4a8aad4d, 4, Finished, Available, Finished, False)